# nnUNet Pipeline — Local GPU Runner

Dataset: **Dataset777_GCEF** | Trainer: **nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss** | Config: **3d_fullres**

Prerequisites (run locally before this notebook):
1. `preprocessing_nnUNet_train.py` → produces `Dataset777_GCEF/` in `nnUNet_raw`
2. (For inference) `preprocessing_nnUNet_predict_tif.py` + `preprocessing_nnUNet_predict_split.py` → produces split chunks

Data is expected at the local paths defined in Section 3.


## 1) Runtime Setup

In [1]:
# Verify GPU runtime
!nvidia-smi

Tue May 19 12:20:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 572.61                 Driver Version: 572.61         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX A6000             WDDM  |   00000000:3B:00.0 Off |                    0 |
| 30%   33C    P8             16W /  300W |    1632MiB /  46068MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os

# Local repository directory
REPO_DIR = r'C:\Users\rony.schwartz\Documents\nnUNet4SoilXrayCT'
assert os.path.isdir(REPO_DIR), f'Repo not found: {REPO_DIR}'
print('Repo dir:', os.listdir(REPO_DIR))


Repo dir: ['.git', '.github', '.vscode', 'analysis', 'colab_nnUNet_pipeline.ipynb', 'dataset_info.json', 'debug_labels_777.py', 'debug_labels_777_followup.py', 'extract_trainlog.py', 'Figures', 'Fiji_macros', 'legacy', 'LICENSE', 'litreture', 'make_annotations.py', 'merge_annotations.py', 'nnUNetTrainer_betterIgnoreSampling.py', 'otsu_threshold_3d.py', 'postprocessing_nnUNet_predict.py', 'postprocessing_nnUNet_predict_concatenate.py', 'postprocessing_pipeline.ipynb', 'preprocess', 'preprocessing_nnUNet_predict.py', 'preprocessing_nnUNet_predict_split.py', 'preprocessing_nnUNet_predict_tif.py', 'preprocessing_nnUNet_train.py', 'preprocess_playground', 'README.md', 'retrieve_dice_score.py', 'run_remaining_fullctx_overnight.ipynb', 'select_slices_and_predict.py', 'setup_prompt.md', 'training_diag', 'Utilities', '__path__.py', '__pycache__']


## 2) Register Custom Trainer

In [10]:
import shutil
import nnunetv2
import os

# Find nnunetv2 trainers directory
nnunet_trainers_dir = os.path.join(
    os.path.dirname(nnunetv2.__file__),
    'training', 'nnUNetTrainer', 'variants', 'sampling'
)
os.makedirs(nnunet_trainers_dir, exist_ok=True)

src = os.path.join(REPO_DIR, 'nnUNetTrainer_betterIgnoreSampling.py')
dst = os.path.join(nnunet_trainers_dir, 'nnUNetTrainer_betterIgnoreSampling.py')
shutil.copy2(src, dst)

# Verify import of early-stopping trainer
from nnunetv2.training.nnUNetTrainer.variants.sampling.nnUNetTrainer_betterIgnoreSampling import nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss
print('Custom trainer registered:', nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss.__name__)

ImportError: cannot import name 'nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss' from 'nnunetv2.training.nnUNetTrainer.variants.sampling.nnUNetTrainer_betterIgnoreSampling' (c:\Users\rony.schwartz\.conda\envs\venv-napari\Lib\site-packages\nnunetv2\training\nnUNetTrainer\variants\sampling\nnUNetTrainer_betterIgnoreSampling.py)

## 3) Set Environment Variables & Paths

In [ ]:
import os

# Local workspace base
LOCAL_BASE = r'C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem'

# Trainer to use across train/inference path construction
TRAINER_NAME = 'nnUNetTrainer_betterIgnoreSampling_earlyStopValLoss'

nnUNet_raw          = os.path.join(LOCAL_BASE, 'nnUNet_raw')
nnUNet_preprocessed = os.path.join(LOCAL_BASE, 'nnUNet_preprocessed')
nnUNet_results      = os.path.join(LOCAL_BASE, 'nnUNet_results')

os.environ['nnUNet_raw']          = nnUNet_raw
os.environ['nnUNet_preprocessed'] = nnUNet_preprocessed
os.environ['nnUNet_results']      = nnUNet_results

# Disable torch.compile — Triton is not available on Windows
os.environ['nnUNet_compile'] = 'false'

for d in [nnUNet_raw, nnUNet_preprocessed, nnUNet_results]:
    os.makedirs(d, exist_ok=True)

print('nnUNet_raw:',          nnUNet_raw)
print('nnUNet_preprocessed:', nnUNet_preprocessed)
print('nnUNet_results:',      nnUNet_results)
print('nnUNet_compile:',      os.environ['nnUNet_compile'])
print('TRAINER_NAME:',        TRAINER_NAME)


nnUNet_raw: C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem\nnUNet_raw
nnUNet_preprocessed: C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem\nnUNet_preprocessed
nnUNet_results: C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem\nnUNet_results
nnUNet_compile: false


## 4) Verify Training Data

The local step `preprocessing_nnUNet_train.py` must have produced:
```
C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem\nnUNet_raw\Dataset777_GCEF\
  imagesTr\*_0000.nii.gz
  labelsTr\*.nii.gz
  dataset.json
```


In [5]:
import subprocess, sys, os

# Run preprocessing_nnUNet_train.py to produce nnUNet_raw/Dataset777_GCEF/
# (imagesTr, labelsTr, dataset.json)
# Skip if already done.
dataset_raw_dir = os.path.join(nnUNet_raw, 'Dataset777_GCEF')
if os.path.isdir(os.path.join(dataset_raw_dir, 'imagesTr')):
    print('Dataset already prepared, skipping preprocessing.')
else:
    print('Running preprocessing_nnUNet_train.py ...')
    result = subprocess.run(
        [sys.executable, os.path.join(REPO_DIR, 'preprocessing_nnUNet_train.py')],
        cwd=REPO_DIR,
        env=os.environ,
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.returncode != 0:
        print("STDERR:\n", result.stderr)
        raise RuntimeError(f"preprocessing_nnUNet_train.py failed (exit {result.returncode})")
    print('Preprocessing done.')


Dataset already prepared, skipping preprocessing.


In [14]:
# Verify uploaded dataset
dataset_dir = os.path.join(nnUNet_raw, 'Dataset777_GCEF')
assert os.path.isdir(dataset_dir), f'Dataset folder not found: {dataset_dir}'

import glob
images = glob.glob(os.path.join(dataset_dir, 'imagesTr', '*_0000.nii.gz'))
labels = glob.glob(os.path.join(dataset_dir, 'labelsTr', '*.nii.gz'))
dataset_json = os.path.join(dataset_dir, 'dataset.json')

print(f'imagesTr: {len(images)} files')
print(f'labelsTr: {len(labels)} files')
print(f'dataset.json exists: {os.path.isfile(dataset_json)}')
assert len(images) > 0, 'No training images found'
assert len(labels) > 0, 'No training labels found'
assert os.path.isfile(dataset_json), 'dataset.json missing'

imagesTr: 1 files
labelsTr: 1 files
dataset.json exists: True


## 5) nnUNet Planning & Preprocessing

In [15]:
import subprocess, sys, os

proc = subprocess.Popen(
    [sys.executable, '-m', 'nnunetv2.experiment_planning.plan_and_preprocess_entrypoints',
     '-d', '777', '--verify_dataset_integrity'],
    env=os.environ,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"plan_and_preprocess failed (exit {proc.returncode})")


Fingerprint extraction...
Dataset777_GCEF
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> as reader/writer

100%|██████████| 1/1 [00:42<00:00, 42.72s/it]
Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [1.03 1.03 1.03]. 
Current patch size: (np.int64(128), np.int64(128), np.int64(128)). 
Current median shape: [632.03883495 631.06796117 631.06796117]
Attempting to find 3d_lowres config. 
Current spacing: [1.0609 1.0609 1.0609]. 

## 6) Training

Run one fold at a time. Change `FOLD` to train multiple folds sequentially (0–4).

**Warning**: each fold may take many hours on a single Colab GPU.

In [16]:
# PolyLRScheduler compatibility fix is applied directly to polylr.py on disk.
# No runtime patch needed here.
print("PolyLRScheduler: using patched polylr.py (PyTorch 2.x compatible)")


PolyLRScheduler: using patched polylr.py (PyTorch 2.x compatible)


In [17]:
# Create custom splits_final.json (single sample used for both train and val)
import json, os

preprocessed_dir = os.path.join(os.environ['nnUNet_preprocessed'], 'Dataset777_GCEF')
splits = [{"train": ["nlm_volume"], "val": ["nlm_volume"]}]

splits_path = os.path.join(preprocessed_dir, 'splits_final.json')
with open(splits_path, 'w') as f:
    json.dump(splits, f, indent=2)

print(f"Wrote {splits_path}")
print(json.dumps(splits, indent=2))

Wrote C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem\nnUNet_preprocessed\Dataset777_GCEF\splits_final.json
[
  {
    "train": [
      "nlm_volume"
    ],
    "val": [
      "nlm_volume"
    ]
  }
]


In [ ]:
import subprocess, sys, os

# Early stopping controls (env-driven in trainer)
os.environ['NNUNET_EARLY_STOP_ENABLED'] = '1'
os.environ['NNUNET_EARLY_STOP_PATIENCE'] = '20'
os.environ['NNUNET_EARLY_STOP_MIN_DELTA'] = '0.001'
print('Early stopping env:', {
    'NNUNET_EARLY_STOP_ENABLED': os.environ['NNUNET_EARLY_STOP_ENABLED'],
    'NNUNET_EARLY_STOP_PATIENCE': os.environ['NNUNET_EARLY_STOP_PATIENCE'],
    'NNUNET_EARLY_STOP_MIN_DELTA': os.environ['NNUNET_EARLY_STOP_MIN_DELTA'],
})

# Stream training output live to the notebook cell
proc = subprocess.Popen(
    [sys.executable, '-m', 'nnunetv2.run.run_training',
     '777', '3d_fullres', '0', '-tr', TRAINER_NAME],
    env=os.environ,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,  # merge stderr into stdout
    text=True,
    bufsize=1
)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f"run_training failed (exit {proc.returncode})")



############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0
c:\Users\rony.schwartz\.conda\envs\venv-napari\Lib\site-packages\nnunetv2\training\nnUNetTrainer\nnUNetTrainer.py:161: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.grad_scaler = GradScaler() if self.device.type == 'cuda' else None

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#################################################

In [ ]:
# List training results
results_dir = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres'
)
if os.path.isdir(results_dir):
    for item in sorted(os.listdir(results_dir)):
        print(item)
else:
    print(f'Results dir not found yet: {results_dir}')

dataset.json
dataset_fingerprint.json
fold_0
plans.json


## 7) Training Outputs

Results are saved locally at:
```
C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem\nnUNet_results\Dataset777_GCEF\<TRAINER_NAME>__nnUNetPlans__3d_fullres\
```
Use `extract_trainlog.py` locally to parse training logs and plot loss curves.


In [ ]:
# Optional: list training result files
results_dir = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres'
)
if os.path.isdir(results_dir):
    for item in sorted(os.listdir(results_dir)):
        print(item)
else:
    print(f'Results dir not found yet: {results_dir}')


dataset.json
dataset_fingerprint.json
fold_0
plans.json


## 8) Inference Data

Run locally first if not done:
1. `preprocessing_nnUNet_predict_tif.py` → `*_0000.nii.gz`
2. `preprocessing_nnUNet_predict_split.py` → split chunks `sample__axis__min__max__0000.nii.gz`

Place split chunks in `C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem\inference_input\` (or update `INFERENCE_INPUT` below).


In [ ]:
import subprocess, sys, os, glob
import numpy as np
import tifffile
import nibabel as nib
from tqdm import tqdm

# === Edit this path to point to the folder with raw .tif volume(s) ===
TIF_INPUT = r'C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem\tif_input'

# Intermediate folder: _0000.nii.gz (skips Fiji — converts .tif directly)
NIFTI_DIR = os.path.join(LOCAL_BASE, 'nifti_predict')
os.makedirs(NIFTI_DIR, exist_ok=True)

# Output folder for split inference chunks
INFERENCE_INPUT = os.path.join(LOCAL_BASE, 'inference_input')
os.makedirs(INFERENCE_INPUT, exist_ok=True)

# Model folder for plans.json
MODEL_DIR = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres'
)

# --- Step 1: .tif -> _0000.nii.gz (direct, no Fiji required) ---
print('=== Step 1: .tif -> _0000.nii.gz (direct Python, zscore norm) ===')
tif_files = glob.glob(os.path.join(TIF_INPUT, '*.tif')) + glob.glob(os.path.join(TIF_INPUT, '*.tiff'))
if not tif_files:
    raise FileNotFoundError(f'No .tif files found in: {TIF_INPUT}')

for tif_path in tqdm(tif_files, desc='Converting TIF'):
    vol = tifffile.imread(tif_path).astype(np.float32)
    # zscore normalization
    mean, std = vol.mean(), vol.std()
    vol = (vol - mean) / (std + 1e-8)
    # nibabel expects (X, Y, Z); tifffile loads (Z, Y, X) — transpose
    vol = vol.transpose(2, 1, 0)
    stem = os.path.splitext(os.path.basename(tif_path))[0]
    out_path = os.path.join(NIFTI_DIR, f'{stem}_0000.nii.gz')
    nib.save(nib.Nifti1Image(vol, affine=np.eye(4)), out_path)
    print(f'  Saved: {out_path}  shape={vol.shape}')

print('Step 1 done.')

# --- Step 2: _0000.nii.gz -> split chunks -> INFERENCE_INPUT ---
print('=== Step 2: preprocessing_nnUNet_predict_split.py ===')
result = subprocess.run(
    [sys.executable,
     os.path.join(REPO_DIR, 'preprocessing_nnUNet_predict_split.py'),
     '-i', NIFTI_DIR,
     '-o', INFERENCE_INPUT,
     '-m', MODEL_DIR],
    env=os.environ, capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:\n', result.stderr)
    raise RuntimeError(f'preprocessing_nnUNet_predict_split.py failed (exit {result.returncode})')
print('Step 2 done.')


=== Step 1: .tif -> _0000.nii.gz (direct Python, zscore norm) ===


Converting TIF: 100%|██████████| 1/1 [01:02<00:00, 62.88s/it]

  Saved: C:\Users\rony.schwartz\Documents\nnUNet_resources\bnei_reem\nifti_predict\nlm_volume_0000.nii.gz  shape=(650, 650, 652)
Step 1 done.
=== Step 2: preprocessing_nnUNet_predict_split.py ===


Model Parameters:
Patch Size:     [128, 128, 128]
Target Spacing: [1.0, 1.0, 1.0]
----------
1 Images found
----------
Image File:    nlm_volume_0000.nii.gz
Image Shape:   (650, 650, 652)
Image Spacing: (1.0, 1.0, 1.0)
Original Patch Size:   [128. 128. 128.]
Patch Overlap:   [64. 64. 64.]
Base Crop Size:   [650 650  82]
 - Split 0 from [0 0 0] to [650 650 146]
 - Split 1 from [ 0  0 18] to [650 650 228]
 - Split 2 from [  0   0 100] to [650 650 310]
 - Split 3 from [  0   0 182] to [650 650 392]
 - Split 4 from [  0   0 264] to [650 650 474]
 - Split 5 from [  0   0 346] to [650 650 556]
 - Split 6 from [  0   0 428] to [650 650 638]
 - Split 7 from [  0   0 510] to [650 650 652]

Step 2 done.


In [8]:
# === Edit these paths to match your local inference data ===
INFERENCE_INPUT  = os.path.join(LOCAL_BASE, 'inference_input')   # folder with split *_0000.nii.gz chunks
INFERENCE_OUTPUT = os.path.join(LOCAL_BASE, 'inference_output')  # output predictions
os.makedirs(INFERENCE_OUTPUT, exist_ok=True)

assert os.path.isdir(INFERENCE_INPUT), f'Inference input not found: {INFERENCE_INPUT}'
input_files = [f for f in os.listdir(INFERENCE_INPUT) if f.endswith('_0000.nii.gz')]
print(f'Inference chunks found: {len(input_files)}')


Inference chunks found: 16


## 9) Inference

In [9]:
import os

# Diagnose: what datasets exist in nnUNet_results?
print('=== nnUNet_results contents ===')
if os.path.isdir(nnUNet_results):
    datasets = os.listdir(nnUNet_results)
    if datasets:
        for d in sorted(datasets):
            print(f'  {d}')
            sub = os.path.join(nnUNet_results, d)
            for item in sorted(os.listdir(sub)):
                print(f'    {item}')
    else:
        print('  (empty — no training results found)')
else:
    print(f'  nnUNet_results dir does not exist: {nnUNet_results}')

# Check specifically for the expected fold checkpoint
expected = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres',
    'fold_0', 'checkpoint_final.pth'
)
print(f'\nExpected checkpoint exists: {os.path.isfile(expected)}')
print(f'Expected checkpoint path:   {expected}')


=== nnUNet_results contents ===
  Dataset777_GCEF
    nnUNetTrainer_betterIgnoreSampling__nnUNetPlans__3d_fullres


NameError: name 'TRAINER_NAME' is not defined

In [ ]:
import torch
import numpy
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor

# nnUNet checkpoints were saved with PyTorch <2.6, which used weights_only=False.
# PyTorch 2.6 changed the default to True, breaking checkpoint loading.
# Patch torch.load to restore weights_only=False (safe: this is our own trained model).
_orig_torch_load = torch.load
def _patched_load(f, *args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _orig_torch_load(f, *args, **kwargs)
torch.load = _patched_load

MODEL_DIR = os.path.join(
    nnUNet_results, 'Dataset777_GCEF',
    f'{TRAINER_NAME}__nnUNetPlans__3d_fullres'
)
print(f'Model dir: {MODEL_DIR}')
print(f'CUDA available: {torch.cuda.is_available()}')

# Auto-select checkpoint: prefer final, fall back to best, then latest
fold_dir = os.path.join(MODEL_DIR, 'fold_0')
for candidate in ('checkpoint_final.pth', 'checkpoint_best.pth', 'checkpoint_latest.pth'):
    if os.path.isfile(os.path.join(fold_dir, candidate)):
        checkpoint_name = candidate
        break
else:
    raise FileNotFoundError(f'No checkpoint found in {fold_dir}')
print(f'Using checkpoint: {checkpoint_name}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

predictor = nnUNetPredictor(
    tile_step_size=0.5,
    use_gaussian=True,
    use_mirroring=True,
    perform_everything_on_device=True,
    device=device,
    verbose=True,
    allow_tqdm=True
)

predictor.initialize_from_trained_model_folder(
    MODEL_DIR,
    use_folds=(0,),
    checkpoint_name=checkpoint_name
)

predictor.predict_from_files(
    INFERENCE_INPUT,
    INFERENCE_OUTPUT,
    save_probabilities=False,
    overwrite=True,
    num_processes_preprocessing=2,
    num_processes_segmentation_export=2,
    folder_with_segs_from_prev_stage=None,
    num_parts=1,
    part_id=0
)
print('Inference complete.')


: 

## 10) Validate Predictions


In [ ]:
# List prediction outputs
pred_files = [f for f in os.listdir(INFERENCE_OUTPUT) if f.endswith('.nii.gz')]
print(f'Predictions: {len(pred_files)} files')
for f in sorted(pred_files):
    print(f'  {f}')